In [4]:
from google.cloud import storage

import os
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.optim as optim
import torch.nn.functional as F

from torchvision import models
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
def unzipArchive():
  client = storage.Client()

  bucketName = 'hammdata'
  bucket = client.get_bucket(bucketName)

  zipBlob = bucket.blob('archive.zip')
  zipBlob.download_to_filename('archive.zip')

  with zipfile.ZipFile('archive.zip', 'r') as zipRef:
      zipRef.extractall('archive')

In [6]:
def saveModel(model, filename):
  client = storage.Client()

  bucketName = 'hammdata'
  bucket = client.get_bucket(bucketName)

  torch.save(model.state_dict(), filename)

  blob = bucket.blob('models/' + filename)
  blob.upload_from_filename(filename)

In [7]:
class HAMDataset(Dataset):
    def __init__(self, data, meta_array, base_path='archive', label_col='target',img_col='image_id', transform=None):
        self.data = data
        self.labelCol = label_col
        self.imgCol = img_col
        self.meta_array = meta_array
        self.transform = transform
        self.basePath = base_path
        self.classes = ["nv", "bkl", "df", "vasc", "mel", "bcc", "akiec"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        imageId = self.data.iloc[idx][self.imgCol]

        try:
            image = Image.open(self.basePath + "/HAM10000_images_part_1/" + imageId + ".jpg")
        except FileNotFoundError:
            try:
                image = Image.open(self.basePath + "/HAM10000_images_part_2/" + imageId + ".jpg")
            except FileExistsError:
                print("Could not find image for: ", imageId)

        label = torch.tensor(self.data.iloc[idx][self.labelCol])
        meta_features = torch.tensor(self.meta_array[idx], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, meta_features, label

In [8]:
class ResNetWithMetadata(nn.Module):
    def __init__(self, num_metadata_features, num_classes):
        super(ResNetWithMetadata, self).__init__()
        self.resnet = models.resnet101(pretrained=True)

        # Replace ResNet's final layer with identity
        self.resnet.fc = nn.Identity()

        # MLP for metadata
        self.meta_fc = nn.Sequential(
            nn.Linear(num_metadata_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
        )

        # Final classifier that combines image and metadata features
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, image, metadata):
        image_features = self.resnet(image)           # [batch, 2048]
        metadata_features = self.meta_fc(metadata)    # [batch, 32]
        combined = torch.cat((image_features, metadata_features), dim=1)
        output = self.classifier(combined)
        return output

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha  # Tensor of shape (num_classes,)
        self.reduction = reduction

    def forward(self, inputs, targets):
        CE_loss = F.cross_entropy(inputs, targets, reduction='none')  # shape (batch,)
        pt = torch.exp(-CE_loss)  # prevents nans when probability is 0

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * (1 - pt) ** self.gamma * CE_loss
        else:
            focal_loss = (1 - pt) ** self.gamma * CE_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [10]:
unzipArchive()

In [11]:
EPOCHS = 50
imgSize = (224, 224)
lr = 5e-5
batchSize = 32
numClasses = 7
gamma = 2.0
alpha = torch.tensor([0.2, 0.9, 1.0, 1.2, 1.0, 1.0, 1.0])  # Example only

binary_map = {
  "nv": 0,
  "bkl": 1,
  "df": 2,
  "vasc": 3,
  "mel": 4,
  "bcc":  5,
  "akiec":  6
}

baseDir = 'archive'
normMeans = [0.76375736, 0.54612675, 0.57056087]
normStd = [0.13945314, 0.1524503,  0.16964808]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

bestAcc = 0
bestModel = None

Using device: cuda


In [12]:
  # read metadata
  metadata = pd.read_csv(baseDir + '/HAM10000_metadata.csv')
  metadata["target"] = metadata["dx"].map(binary_map)
  # metadata = metadata.groupby('lesion_id').sample(n=1, random_state=42).reset_index(drop=True)

  metadata['age'] = metadata['age'].fillna(metadata['age'].median())
  metadata['sex'] = metadata['sex'].fillna('unknown')
  metadata['localization'] = metadata['localization'].fillna('unknown')

  scaler = StandardScaler()
  metadata['age'] = scaler.fit_transform(metadata[['age']])

In [13]:
transform = transforms.Compose([transforms.Resize(imgSize),
                                transforms.RandomHorizontalFlip(),
                                transforms.RandomVerticalFlip(),
                                transforms.RandomRotation(20),
                                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                                transforms.ToTensor(),
                                transforms.Normalize(normMeans, normStd)])

valTestTransform = transforms.Compose([transforms.Resize(imgSize),
                                          transforms.ToTensor(),
                                          transforms.Normalize(normMeans, normStd)])

In [14]:
train_df, temp_df = train_test_split(
    metadata, test_size=0.3, stratify=metadata['target'], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['target'], random_state=42
)

train_meta_encoded = train_df[['age', 'sex', 'localization']].copy()
train_encoded = pd.get_dummies(train_meta_encoded, columns=['sex', 'localization'])
train_meta = torch.tensor(train_encoded.loc[train_df.index].to_numpy().astype(np.float32))

val_meta_encoded = val_df[['age', 'sex', 'localization']].copy()
val_encoded = pd.get_dummies(val_meta_encoded, columns=['sex', 'localization'])
for index, dumCol in enumerate(train_encoded.columns):
    if dumCol not in val_encoded:
        val_encoded.insert(loc=index, column=dumCol, value=False)

val_meta   = torch.tensor(val_encoded.loc[val_df.index].to_numpy().astype(np.float32))

test_meta_encoded = test_df[['age', 'sex', 'localization']].copy()
test_encoded = pd.get_dummies(test_meta_encoded, columns=['sex', 'localization'])
for index, dumCol in enumerate(train_encoded.columns):
    if dumCol not in test_encoded:
        test_encoded.insert(loc=index, column=dumCol, value=False)

test_meta  = torch.tensor(test_encoded.loc[test_df.index].to_numpy().astype(np.float32))


In [15]:
train_dataset = HAMDataset(train_df, train_meta, transform=transform)
val_dataset   = HAMDataset(val_df, val_meta, transform=valTestTransform)
test_dataset  = HAMDataset(test_df, test_meta, transform=valTestTransform)

train_loader = DataLoader(train_dataset, batch_size=batchSize, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batchSize, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batchSize, shuffle=False)

In [16]:
num_metadata_features = train_meta.shape[1]

model = ResNetWithMetadata(num_metadata_features=num_metadata_features,
                        num_classes=numClasses).to(device)

criterion = nn.CrossEntropyLoss()
# criterion = FocalLoss(gamma=gamma, alpha=alpha.to(device))
optimizer = optim.Adam(model.parameters(), lr=lr)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100%|██████████| 171M/171M [00:01<00:00, 123MB/s]


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, metadata, labels in loop:
        images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images, metadata)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(loss=loss.item(), acc=100 * correct / total)

    # scheduler.step()
    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | Accuracy: {100 * correct / total:.2f}%")

    # 🎯 Training summary
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    print(f"✅ Training Loss: {train_loss:.4f} | Accuracy: {train_acc:.2f}%")

    # 🧪 VALIDATION
    # val_preds, val_labels = evaluate_model(model, valLoader, device)

    model.eval()
    valCorrect = 0
    valTotal = 0

    with torch.no_grad():
        for images, metadata, labels in val_loader:
            images, metadata, labels = images.to(device), metadata.to(device), labels.to(device)
            outputs = model(images, metadata)
            _, predicted = torch.max(outputs, 1)
            valTotal += labels.size(0)
            valCorrect += (predicted == labels).sum().item()

    valAcc = 100 * valCorrect / valTotal
    print(f"Validation Accuracy: {100 * valCorrect / valTotal:.2f}%")

    # 💾 Save best model by F1
    if valAcc > bestAcc:
        bestAcc = valAcc
        bestModel = model
        torch.save(model.state_dict(), "best_val1_dup_model.pt")
        print(f"💾 Saved new best model")

Epoch 1/50:   0%|          | 0/220 [00:00<?, ?it/s]<ipython-input-7-25caff51c8a9>:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  meta_features = torch.tensor(self.meta_array[idx], dtype=torch.float32)
Epoch 1/50: 100%|██████████| 220/220 [02:12<00:00,  1.66it/s, acc=72, loss=0.514]


Epoch 1 | Loss: 0.8297 | Accuracy: 72.03%
✅ Training Loss: 0.8297 | Accuracy: 72.03%
Validation Accuracy: 75.70%
💾 Saved new best model


Epoch 2/50:  60%|█████▉    | 131/220 [01:17<00:52,  1.71it/s, acc=79.6, loss=0.624]

In [ ]:
def evaluate_model(model, dataloader, device):
    model.eval()
    total = 0
    correct = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, metadata, labels in dataloader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            outputs = model(images, metadata)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(probs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    print(f"✅ Eval Accuracy: {accuracy:.2f}%")
    return all_preds, all_labels, all_probs

In [ ]:
  # After training is done:
  valPreds, valLabels, valProbs = evaluate_model(bestModel, val_loader, device)

In [ ]:
    testPreds, testLabels, testProbs = evaluate_model(bestModel, test_loader, device)

In [ ]:
    # Print classification report
    print(classification_report(valLabels, valPreds, target_names=train_dataset.classes))
    print(classification_report(testPreds, testLabels, target_names=train_dataset.classes))

In [ ]:
    # Confusion matrix
    cm = confusion_matrix(valLabels, valPreds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=train_dataset.classes, yticklabels=train_dataset.classes, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Val Confusion Matrix')
    plt.show()

In [ ]:
saveModel(bestModel, 'best_val_meta_dup_model_89.pt')

In [ ]:
    cm = confusion_matrix(testLabels, testPreds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=train_dataset.classes, yticklabels=train_dataset.classes, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Confusion Matrix')
    plt.show()